## Scanrefer Dataset Class

In [ ]:
import torch
import json
import numpy as np
import trimesh # Good for loading .ply meshes
from torch.utils.data import Dataset

class ScanReferDataset(Dataset):
    def __init__(self, scanrefer_data_path, scannet_dir, num_points=40000):
        # 1. Load the ScanRefer JSON (the snippet you provided)
        with open(scanrefer_data_path, 'r') as f:
            self.scanrefer_data = json.load(f)
            
        self.scannet_dir = scannet_dir
        self.num_points = num_points

    def __len__(self):
        return len(self.scanrefer_data)

    def __getitem__(self, idx):
        # 1. Get the specific query data
        item = self.scanrefer_data[idx]
        scene_id = item["scene_id"]
        target_obj_id = int(item["object_id"])
        
        # 2. Load the 3D Point Cloud (.ply)
        ply_path = f"{self.scannet_dir}/{scene_id}/{scene_id}_vh_clean_2.ply"
        mesh = trimesh.load(ply_path, process=False)
        points = np.array(mesh.vertices) # (N, 3) XYZ coordinates
        colors = np.array(mesh.visual.vertex_colors[:, :3]) # (N, 3) RGB values
        
        # Combine XYZ and RGB
        point_cloud = np.concatenate([points, colors], axis=1)
        
        # 3. Downsample to self.num_points
        # (For initial testing, random choice is faster than Farthest Point Sampling)
        choices = np.random.choice(point_cloud.shape[0], self.num_points, replace=False)
        point_cloud = point_cloud[choices, :]
        
        # 4. Extract Ground Truth (Pseudocode)
        # -> Load .aggregation.json and .segs.json
        # -> Find which points correspond to target_obj_id
        # -> Calculate bounding box center and size from those points
        gt_box_center = np.array([0.0, 0.0, 0.0]) # Replace with actual parsing logic
        gt_box_size = np.array([1.0, 1.0, 1.0])   # Replace with actual parsing logic
        
        # 5. Get the Text Tokens
        # (Later, we will pass this through RoBERTa's tokenizer)
        text_tokens = item["token"]
        
        return {
            "point_cloud": torch.tensor(point_cloud, dtype=torch.float32),
            "gt_box_center": torch.tensor(gt_box_center, dtype=torch.float32),
            "gt_box_size": torch.tensor(gt_box_size, dtype=torch.float32),
            "text": " ".join(text_tokens)
        }

## Test Data Pipeline

## Text Encoder

In [5]:
import torch
import torch.nn as nn
from transformers import RobertaTokenizer, RobertaModel

class TextEncoder(nn.Module):
    def __init__(self, model_name="FacebookAI/roberta-base"):
        super().__init__()
        self.tokenizer = RobertaTokenizer.from_pretrained(model_name)
        # Load the base model (no classification head attached)
        self.roberta = RobertaModel.from_pretrained(model_name)
        
        # Optional: Freeze the RoBERTa weights initially to train the 3D backbone faster
        # for param in self.roberta.parameters():
        #     param.requires_grad = False

    def forward(self, text_list):
        # 1. Tokenize the batch of text queries
        inputs = self.tokenizer(
            text_list, 
            padding=True, 
            truncation=True, 
            max_length=80, # ScanRefer descriptions rarely exceed this
            return_tensors="pt"
        )
        
        # Move inputs to the same device as the model
        inputs = {k: v.to(self.roberta.device) for k, v in inputs.items()}
        
        # 2. Pass through RoBERTa
        outputs = self.roberta(**inputs)
        
        # 3. Extract the features
        # word_embeddings shape: (Batch_Size, Sequence_Length, 768)
        word_embeddings = outputs.last_hidden_state 
        
        # sentence_embedding shape: (Batch_Size, 768) - Often used for global matching
        sentence_embedding = outputs.pooler_output 
        
        return word_embeddings, sentence_embedding, inputs['attention_mask']

## Multimodal Fusion Model

In [4]:
import torch
import torch.nn as nn

class CrossAttentionFusion(nn.Module):
    def __init__(self, text_dim=768, vision_dim=256, hidden_dim=256, num_heads=4):
        super().__init__()
        
        # 1. Dimensionality Alignment
        # RoBERTa outputs 768-dim, PointNet usually outputs 256-dim. 
        # They must be the same size to interact in the attention mechanism.
        self.text_proj = nn.Linear(text_dim, hidden_dim)
        self.vision_proj = nn.Linear(vision_dim, hidden_dim)
        
        # 2. The Cross-Attention Layer
        # batch_first=True makes tensor shapes easier to manage (B, N, C)
        self.cross_attn = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=num_heads, batch_first=True)
        
        # 3. The Grounding Predictor
        # A small MLP to take the fused feature and output a single "Match Score"
        self.match_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1) # Outputs a single score per bounding box
        )

    def forward(self, text_features, box_features, text_mask=None):
        # text_features: (Batch, Seq_Len, 768)
        # box_features: (Batch, Num_Boxes, 256)
        
        # 1. Project to common hidden dimension
        text_emb = self.text_proj(text_features) # (Batch, Seq_Len, 256)
        box_emb = self.vision_proj(box_features) # (Batch, Num_Boxes, 256)
        
        # 2. Cross-Attention
        # Queries: Box Features (The boxes are asking "Do I match the text?")
        # Keys/Values: Text Features (The text provides the answers)
        # We also pass the RoBERTa attention mask so the network ignores padding tokens
        fused_features, _ = self.cross_attn(
            query=box_emb, 
            key=text_emb, 
            value=text_emb, 
            key_padding_mask=(~text_mask.bool()) if text_mask is not None else None
        )
        
        # fused_features shape: (Batch, Num_Boxes, 256)
        
        # 3. Predict Grounding Scores
        # Squeeze removes the last dimension to make it (Batch, Num_Boxes)
        match_scores = self.match_head(fused_features).squeeze(-1) 
        
        return match_scores

## Loss

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GroundingLoss3D(nn.Module):
    def __init__(self, box_weight=1.0, match_weight=0.1):
        super().__init__()
        self.box_weight = box_weight
        self.match_weight = match_weight
        self.smooth_l1 = nn.SmoothL1Loss()
        # Cross Entropy for matching text to the correct box
        self.match_loss = nn.CrossEntropyLoss() 

    def forward(self, pred_centers, pred_sizes, match_scores, gt_centers, gt_sizes, target_box_indices):
        """
        pred_centers: (Batch, Num_Proposals, 3) - Predicted (cx, cy, cz)
        pred_sizes: (Batch, Num_Proposals, 3) - Predicted (dx, dy, dz)
        match_scores: (Batch, Num_Proposals) - The text-to-box scores from your fusion module
        gt_centers: (Batch, 3) - Ground truth target centers
        gt_sizes: (Batch, 3) - Ground truth target sizes
        target_box_indices: (Batch,) - The index of the proposal closest to the GT box
        """
        batch_size = pred_centers.shape[0]

        # 1. Box Regression Loss (Only calculate for the matched target box)
        # We gather the predicted center and size for the specific box that 
        # the model was *supposed* to match with the text.
        
        batch_indices = torch.arange(batch_size)
        target_pred_centers = pred_centers[batch_indices, target_box_indices]
        target_pred_sizes = pred_sizes[batch_indices, target_box_indices]

        # L1 Loss for Center (cx, cy, cz)
        loss_center = self.smooth_l1(target_pred_centers, gt_centers)
        
        # L1 Loss for Size (dx, dy, dz)
        loss_size = self.smooth_l1(target_pred_sizes, gt_sizes)
        
        loss_box = loss_center + loss_size

        # 2. Grounding Matching Loss
        # We treat visual grounding as a classification problem over the proposals.
        # target_box_indices is the "correct class" (the ID of the correct bounding box).
        loss_match = self.match_loss(match_scores, target_box_indices)

        # 3. Total Loss
        total_loss = (self.box_weight * loss_box) + (self.match_weight * loss_match)

        return total_loss, loss_box, loss_match

In [ ]:
import torch

def inject_votenet_weights(my_grounding_model, weights_path):
    print("Loading pre-trained VoteNet weights...")
    
    # 1. Load the downloaded checkpoint
    # Always map to CPU first to avoid GPU memory spikes during loading
    checkpoint = torch.load(weights_path, map_location='cpu')
    
    # Many standard VoteNet checkpoints nest the weights under 'model_state_dict'
    if 'model_state_dict' in checkpoint:
        pretrained_dict = checkpoint['model_state_dict']
    else:
        pretrained_dict = checkpoint

    # 2. Get the empty state dictionary of your custom model
    model_dict = my_grounding_model.state_dict()

    # 3. The Filtration Process
    filtered_dict = {}
    skipped_keys = []
    
    for key, weight_tensor in pretrained_dict.items():
        # We prepend 'backbone.' or whatever you named your VoteNet module in your __init__
        # If your VoteNet is nested under self.detector, change 'detector.' + key
        
        # Check 1: Does this layer exist in our custom model?
        if key in model_dict:
            # Check 2: Are the tensor shapes identical?
            if weight_tensor.shape == model_dict[key].shape:
                filtered_dict[key] = weight_tensor
            else:
                skipped_keys.append((key, "Shape Mismatch"))
        else:
            # Often, we skip the final semantic classification layers of VoteNet
            skipped_keys.append((key, "Not in Custom Model"))

    # 4. Inject the filtered weights into your model
    model_dict.update(filtered_dict)
    
    # strict=False allows the model to accept the dictionary even if some layers 
    # (like your Cross-Attention fusion module) are left uninitialized.
    my_grounding_model.load_state_dict(model_dict, strict=False)

    # 5. Terminal Feedback
    print(f"Successfully injected {len(filtered_dict)} layers!")
    print(f"Skipped {len(skipped_keys)} layers (usually final classification heads).")
    
    return my_grounding_model

In [ ]:
# 1. Instantiate your custom architecture
vg_model = VisualGrounding3D()

# 2. Inject the weights
votenet_weights_path = "./votenet_scannet_pretrained.pth" # Path to your download
vg_model = inject_votenet_weights(vg_model, votenet_weights_path)

# Move model to GPU after loading weights
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vg_model.to(device)

## Train Config

In [ ]:
import torch.optim as optim

# 1. Define the specific learning rates
# Backbones get a tiny learning rate to preserve their pre-trained knowledge
backbone_lr = 1e-5  
text_lr = 1e-5      

# The fusion module gets a standard learning rate because it is learning from scratch
fusion_lr = 1e-3    

# 2. Group the parameters
# Note: Ensure the variable names match how you defined them in your __init__
param_groups = [
    {'params': model.backbone.parameters(), 'lr': backbone_lr},
    {'params': model.vgen.parameters(), 'lr': backbone_lr},
    {'params': model.pnet.parameters(), 'lr': backbone_lr},
    {'params': model.text_encoder.parameters(), 'lr': text_lr},
    
    # The new multimodal layers
    {'params': model.cross_attn.parameters(), 'lr': fusion_lr},
    {'params': model.match_head.parameters(), 'lr': fusion_lr} 
]

print("Trainable Layers:")
for name, param in vg_model.named_parameters(): 
    if param.requires_grad:
        print(f"+ {name}")

# 4. Print the summary formatted in Millions (1e6)
print("\n--- Parameter Summary ---")
print(f"Total params: {total_params_count / 1_000_000:.2f} M")
print(f"Trainable params: {trainable_params_count / 1_000_000:.2f} M")
print(f"Percentage trainable: {trainable_percentage:.2f} %")
print("-------------------------\n")


# 3. Initialize the Optimizer
# AdamW is highly recommended when working with Transformer architectures (RoBERTa)
optimizer = optim.AdamW(param_groups, weight_decay=1e-4)

# Initialize your custom loss function
criterion = GroundingLoss3D(box_weight=1.0, match_weight=0.1)

## Training

## Validation

In [ ]:
import torch

def calculate_3d_iou_aabb(pred_centers, pred_sizes, gt_centers, gt_sizes):
    """
    Calculates 3D IoU for Axis-Aligned Bounding Boxes.
    Expects tensors of shape (Batch_Size, 3) for centers and sizes.
    """
    # 1. Calculate min and max coordinates for Predicted Boxes (x, y, z)
    pred_min = pred_centers - (pred_sizes / 2.0)
    pred_max = pred_centers + (pred_sizes / 2.0)
    
    # 2. Calculate min and max coordinates for Ground Truth Boxes
    gt_min = gt_centers - (gt_sizes / 2.0)
    gt_max = gt_centers + (gt_sizes / 2.0)
    
    # 3. Calculate Intersection volume
    intersect_min = torch.max(pred_min, gt_min)
    intersect_max = torch.min(pred_max, gt_max)
    
    # Ensure no negative dimensions (if boxes don't overlap, intersection is 0)
    intersect_dims = torch.clamp(intersect_max - intersect_min, min=0.0)
    intersect_volume = intersect_dims[:, 0] * intersect_dims[:, 1] * intersect_dims[:, 2]
    
    # 4. Calculate Union volume
    pred_volume = pred_sizes[:, 0] * pred_sizes[:, 1] * pred_sizes[:, 2]
    gt_volume = gt_sizes[:, 0] * gt_sizes[:, 1] * gt_sizes[:, 2]
    union_volume = pred_volume + gt_volume - intersect_volume
    
    # 5. Calculate IoU
    iou = intersect_volume / torch.clamp(union_volume, min=1e-6)
    
    return iou # Shape: (Batch_Size,)

In [ ]:
import os

epochs = 50
best_val_acc_025 = 0.0 # Track the best accuracy to save the optimal model
checkpoint_dir = "./checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

for epoch in range(epochs):
    # ==========================================
    #               TRAINING PHASE
    # ==========================================
    model.train()
    train_loss = 0.0
    
    fprint(f"--- Starting Epoch {epoch+1} ---")
    epoch_loss = 0.0
    
    for batch_idx, batch in enumerate(dataloader):
        # 1. Move everything to the GPU
        points = batch["point_cloud"].to(device)
        gt_centers = batch["gt_box_center"].to(device)
        gt_sizes = batch["gt_box_size"].to(device)
        
        # (Assuming you tokenized the text and moved it to the device)
        text_inputs = batch["text_tokens"].to(device) 
        text_masks = batch["text_masks"].to(device)
        
        # 2. Clear previous gradients
        optimizer.zero_grad()
        
        # 3. Forward Pass
        # Returns coordinates/sizes for 256 boxes, and 256 matching scores
        pred_centers, pred_sizes, match_scores = model(points, text_inputs, text_masks)
        
        # 4. Target Assignment (The greedy matching step)
        # Find which of the 256 proposed centers is geometrically closest to the GT center
        # cdist computes the euclidean distance between batches of points
        distances = torch.cdist(pred_centers, gt_centers.unsqueeze(1)) # Shape: (Batch, 256, 1)
        distances = distances.squeeze(-1) # Shape: (Batch, 256)
        
        # The index of the box with the minimum distance is our target class
        target_box_indices = torch.argmin(distances, dim=1) # Shape: (Batch,)
        
        # 5. Calculate Loss
        total_loss, loss_box, loss_match = criterion(
            pred_centers, 
            pred_sizes, 
            match_scores, 
            gt_centers, 
            gt_sizes, 
            target_box_indices
        )
        
        # 6. Backward Pass
        total_loss.backward()
        
        # Gradient Clipping (Crucial for stability when training transformers + 3D)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        
        # 7. Update Weights
        optimizer.step()
        
        epoch_loss += total_loss.item()
        
        if batch_idx % 10 == 0:
            print(f"Batch {batch_idx} | Total Loss: {total_loss.item():.4f} | "
                  f"Box L1: {loss_box.item():.4f} | Match CE: {loss_match.item():.4f}")

    print(f"Epoch {epoch+1} Complete | Average Loss: {epoch_loss/len(dataloader):.4f}")

    # ==========================================
    #              VALIDATION PHASE
    # ==========================================
    model.eval() # Turn off dropout, freeze batch normalization
    
    total_val_samples = 0
    passed_025 = 0
    passed_050 = 0
    
    # torch.no_grad() prevents PyTorch from storing massive gradient graphs in memory
    with torch.no_grad():
        for batch in val_dataloader:
            points = batch["point_cloud"].to(device)
            gt_centers = batch["gt_box_center"].to(device)
            gt_sizes = batch["gt_box_size"].to(device)
            text_inputs = batch["text_tokens"].to(device) 
            text_masks = batch["text_masks"].to(device)
            
            # 1. Forward Pass
            pred_centers, pred_sizes, match_scores = model(points, text_inputs, text_masks)
            
            # 2. Select the Best Box (The network's final answer)
            # Find the index of the box with the highest matching score
            best_box_indices = torch.argmax(match_scores, dim=1) # Shape: (Batch,)
            
            # Extract the coordinates of the chosen boxes using fancy indexing
            batch_indices = torch.arange(points.shape[0])
            chosen_centers = pred_centers[batch_indices, best_box_indices]
            chosen_sizes = pred_sizes[batch_indices, best_box_indices]
            
            # 3. Calculate 3D IoU
            ious = calculate_3d_iou_aabb(chosen_centers, chosen_sizes, gt_centers, gt_sizes)
            
            # 4. Tally the passes
            total_val_samples += points.shape[0]
            passed_025 += torch.sum(ious >= 0.25).item()
            passed_050 += torch.sum(ious >= 0.50).item()

    # Calculate final epoch percentages
    acc_025 = (passed_025 / total_val_samples) * 100.0
    acc_050 = (passed_050 / total_val_samples) * 100.0
    
    print(f"--- Epoch {epoch+1} Validation ---")
    print(f"Acc@0.25: {acc_025:.2f}% | Acc@0.5: {acc_050:.2f}%")

    # ==========================================
    #              CHECKPOINT SAVING
    # ==========================================
    
    # Create the state dictionary payload
    checkpoint_state = {
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_acc_025': best_val_acc_025
    }

    # 1. Save Best Model
    if acc_025 > best_val_acc_025:
        print(f"🚀 New High Score! Saving best model (Acc@0.25: {acc_025:.2f}%)")
        best_val_acc_025 = acc_025
        best_path = os.path.join(checkpoint_dir, "best_grounding_model.pth")
        torch.save(checkpoint_state, best_path)

    # 2. Save Interval Model (Every 10 epochs)
    if (epoch + 1) % 10 == 0:
        print(f"💾 Saving interval checkpoint at epoch {epoch + 1}...")
        interval_path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch+1}.pth")
        torch.save(checkpoint_state, interval_path)